In [1]:
import pandas as pd
import numpy
import seaborn as sns
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from supabase_helpers import get_from_supabase, get_ta_from_supabase
import os
import dotenv

dotenv.load_dotenv()

df = get_from_supabase(ticker="AAPL", interval="daily")
df_ta = get_ta_from_supabase(ticker="AAPL", interval="daily")

In [2]:
df.head()

,id,ticker,timestamp,open,high,low,close,volume,log_return
0,1,AAPL,2022-03-18,160.509995,164.479996,159.759995,163.979996,123511700,0.020703
1,2,AAPL,2022-03-21,163.509995,166.350006,163.009995,165.380005,95811400,0.008502
2,3,AAPL,2022-03-22,165.509995,169.419998,164.910004,168.820007,81532000,0.020587
3,4,AAPL,2022-03-23,167.990005,172.639999,167.649994,170.210007,98062700,0.008200
4,5,AAPL,2022-03-24,171.059998,174.139999,170.210007,174.070007,90131400,0.022425


In [3]:
df_ta.head()

,id,ticker,sma_50,sma_200,sma_spread,macd_line,macd_signal,macd_hist,rsi_14,bb_upper,bb_lower,bbw,atr_14,hvr,obv,stock_daily_id
0,1,AAPL,142.709599,151.43325,-0.057607,-4.719809,-3.838464,-0.881345,22.617001,149.775583,122.660417,0.199057,4.571426,0.849085,8214729100,200
1,2,AAPL,142.291399,151.24515,-0.059200,-4.800808,-4.030933,-0.769875,23.287651,148.513103,121.895897,0.196866,4.418569,0.877441,8303842700,201
2,3,AAPL,141.802799,151.04335,-0.061178,-4.916453,-4.208037,-0.708416,23.902412,147.840951,120.779048,0.201488,4.240712,0.877614,8303842700,202
3,4,AAPL,141.348399,150.84735,-0.062971,-4.584080,-4.283246,-0.300834,39.492959,147.052434,120.435564,0.199014,4.113568,0.943872,8391597400,203
4,5,AAPL,140.964399,150.64705,-0.064274,-4.229153,-4.272427,0.043274,43.030682,145.826902,120.411096,0.190925,4.104283,0.908133,8462388200,204


In [2]:
df_all = pd.merge(df, df_ta, left_on="id", right_on="stock_daily_id", how="left")
df_all.loc[198]

id_x                     199
ticker_x                AAPL
timestamp         2022-12-30
open              128.410004
high              129.949997
low                   127.43
close             129.929993
volume              77034200
log_return          0.002466
id_y                     NaN
ticker_y                 NaN
sma_50                   NaN
sma_200                  NaN
sma_spread               NaN
macd_line                NaN
macd_signal              NaN
macd_hist                NaN
rsi_14                   NaN
bb_upper                 NaN
bb_lower                 NaN
bbw                      NaN
atr_14                   NaN
hvr                      NaN
obv                      NaN
stock_daily_id           NaN
Name: 198, dtype: object

In [4]:
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    row_heights=[0.5, 0.25, 0.25],
    vertical_spacing=0.03
)

# Row 1 — Price + moving averages (same scale, no secondary needed)
fig.add_trace(go.Scatter(x=df_all["timestamp"], y=df_all["close"], name="Price"), row=1, col=1)
fig.add_trace(go.Scatter(x=df_all["timestamp"], y=df_all["sma_50"], name="SMA 50"), row=1, col=1)
fig.add_trace(go.Scatter(x=df_all["timestamp"], y=df_all["sma_200"], name="SMA 200"), row=1, col=1)

# Row 2 — RSI
fig.add_trace(go.Scatter(x=df_all["timestamp"], y=df_all["rsi_14"], name="RSI"), row=2, col=1)
fig.add_hrect(y0=30, y1=70, fillcolor="gray", opacity=0.1, row=2, col=1)  # overbought/oversold band

# Row 3 — Volume bars
fig.add_trace(go.Bar(x=df_all["timestamp"], y=df_all["volume"], name="Volume"), row=3, col=1)

fig.update_layout(height=700, title="Stock Dashboard", xaxis_rangeslider_visible=False)
fig.show()